# LoRA Training for Intention Identification - Qwen3.5-4B with Unsloth

## 1. Preparation

### 1.1 Import dependencies

In [1]:
import re
import os
import gc
from datetime import datetime
import json
import ast
import re
from tqdm import tqdm
import torch
from datasets import load_dataset, DatasetDict

from unsloth import FastLanguageModel
from transformers import (
    DataCollatorForSeq2Seq,
    Trainer,    
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)
from trl import SFTTrainer

INFO 03-20 19:43:53 __init__.py:183] Automatically detected platform cuda.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


[fla.utils|WARNING]Current Triton version 3.1.0 is below the recommended 3.2.0 version. Errors may occur and these issues will not be fixed. Please consider upgrading Triton.


🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
import warnings
warnings.filterwarnings('ignore')

### 1.2 Set constants

In [3]:
MODEL_NAME = "../models/Qwen/Qwen3___5-4B"
DATA_PATH = "../data/final/LoRA_samples_sum_fixed.jsonl"

OUTPUT_DIR = "./fine_tuning_output"

MAX_LENGTH = 2048
SEED = 42

set_seed(SEED)

Enable TensorFloat-32 (TF32) on CUDA and cuDNN to accelerate matrix and convolution operations with minimal precision loss.

In [4]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

## 2. Dataset

### 2.1 Load and shuffle the dataset

In [5]:
# Load dataset
ds_raw = load_dataset("json", data_files=DATA_PATH)["train"]

In [6]:
print(ds_raw)

Dataset({
    features: ['messages'],
    num_rows: 10104
})


In [7]:
# Shuffle the datasets
shuffled_ds = ds_raw.shuffle(seed=SEED)

Check the shuffled dataset.

In [8]:
shuffled_ds[:3]

{'messages': [[{'role': 'user',
    'content': "## === 你的角色描述和背景信息（仅供参考） ===\n你是一个家装博览会的电话邀约专员，负责邀请南通市民参加本周末的家装建材采购节。\n\n## === 你的核心任务（必须遵守） ===\n你需要根据以下指示判断**最后一次用户输入**的意图，然后**仅输出一个JSON对象**，绝对不能是其他任何格式\n\n### 重要规则：\n1. 你只负责判断意图，**不进行对话**\n2. 无论对话历史如何，你的**输出必须是且仅是一个JSON对象**，不得输出为空、不得输出其他格式的数据\n3. 禁止输出任何自然语言、解释或额外内容\n4. 如果之前的回复是自然语言，忽略它并继续输出JSON\n5. JSON必须包含且仅包含两个字段：`input_summary`、`intention_id`。字段值必须为字符串，格式必须严格遵循：{'input_summary': '...', 'intention_id': '...'}\n\n### JSON字段说明：\n1. **input_summary**\n- 务必参考**智能助手和用户的对话历史**，用不超过10个汉字，清晰简洁地概括**最后一次用户输入**的核心内容\n- 示例：“用户有兴趣参加活动”、“用户不想被打扰”、“用户询问你是谁”\n\n2. **intention_id**\n- 务必参考**智能助手和用户的对话历史**，判断**最后一次用户输入**的意图\n- 仅可从以下【意图库列表】或【知识库列表】中，根据**意图名称**和**意图说明**，选择**唯一最匹配**的意图\n- 若最后一次用户输入无法匹配任一意图，输出 `intention_id` 为 'others'\n- 如果最后一次用户输入的语义匹配某一条意图的**意图名称**和**意图说明**，则将这条意图的**意图id**输出为 `intention_id`\n- 禁止自行创建或修改 `intention_id`，其值必须严格来自两个列表中的意图id，或为 'others'\n- 选择优先级说明：\n  - 如果两个列表均有内容，则同时使用【知识库列表】和【意图库列表】判别用户的意图，不分先后，当匹配成功，将**意图id**输出为 `intenti

Under a dict, there is one key "messages", the value is a list. Each item in the list is a data sample (also a list).

### 2.2 Split the dataset
Split it into train (75%), validation (15%), test (10%)

In [9]:
train_temp = shuffled_ds.train_test_split(test_size=0.25, seed=SEED)

val_test = train_temp['test'].train_test_split(test_size=0.4, seed=42)

ds = DatasetDict({
    'train': train_temp['train'], # 75%
    'validation': val_test['train'], # 15%
    'test': val_test['test'] # 10%
})

Check the split datasets.

In [10]:
ds["train"]

Dataset({
    features: ['messages'],
    num_rows: 7578
})

In [11]:
ds["validation"]

Dataset({
    features: ['messages'],
    num_rows: 1515
})

In [12]:
ds["test"]

Dataset({
    features: ['messages'],
    num_rows: 1011
})

In [13]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 7578
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 1515
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 1011
    })
})


### 2.3 Tokenization

Load the tokenizer, together with the model with FastLanguageModel from Unsloth.

In [14]:
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1" # use local mode, we load the model from local drive
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_LENGTH,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    trust_remote_code=True,
)

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.8: Fast Qwen3_5 patching. Transformers: 5.3.0. vLLM: 0.7.0.
   \\   /|    NVIDIA vGPU-32GB. Num GPUs = 1. Max memory: 31.473 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Qwen3_5 does not support SDPA - switching to fast eager.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [15]:
# Get only the tokenizer for text (this is a multimodal model)
tokenizer = tokenizer.tokenizer

# The model is also multimodel, we freeze the visual part
for param in model.model.visual.parameters():
    param.requires_grad = False

Configure and check the tokenizer. In training, we set the padding on the right to keep the labels aligned, and make it easy to mask loss.

In [16]:
tokenizer.padding_side = "right"
tokenizer.truncation_side = "left"

In [17]:
# The tokenizer already has PAD token and EOS token. There is no need for extra set up.
print(f"\\nTokenizer info:")
print(f"PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

\nTokenizer info:
PAD token: <|endoftext|> (ID: 248044)
EOS token: <|im_end|> (ID: 248046)


Set up the process function for correct tokenization for the data samples.

In [18]:
def process_func(example):

    messages = example["messages"]

    instruction_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    instruction_text = re.sub(r'\n<think>\s*</think>\n', '', instruction_text)

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    ) + tokenizer.eos_token
    full_text = re.sub(r'\n<think>\s*</think>\n', '', full_text)

    tokenized_full = tokenizer( 
        text=full_text,
        max_length=MAX_LENGTH,
        truncation=True,
        add_special_tokens=False
    )

    tokenized_instruction = tokenizer(
        text=instruction_text,
        max_length=MAX_LENGTH,
        truncation=True,
        add_special_tokens=False
    )

    input_ids = tokenized_full["input_ids"]
    attention_mask = tokenized_full["attention_mask"]

    instruction_len = len(tokenized_instruction["input_ids"])

    labels = [-100] * instruction_len + input_ids[instruction_len:]
    labels = labels[:len(input_ids)]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

Tokenize, and check the tokenized data.

In [19]:
tokenized_ds = ds.map(
    process_func,
    remove_columns=ds["train"].column_names,
    num_proc=os.cpu_count(),
    desc="Processing dataset"
)

In [20]:
tokenized_ds

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 7578
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1515
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1011
    })
})

In [21]:
len(tokenized_ds["train"]["input_ids"][5])

1387

In [22]:
# Check sequence lengths
max_len = max(len(x) for x in tokenized_ds["train"]["input_ids"])
avg_len = sum(len(x) for x in tokenized_ds["train"]["input_ids"]) / len(tokenized_ds["train"])
print(f"Sequence length - Max: {max_len}, Avg: {avg_len:.1f}")

Sequence length - Max: 1954, Avg: 1454.7


In [23]:
tokenized_ds["train"][2]

{'input_ids': [248045,
  846,
  198,
  550,
  1981,
  220,
  97319,
  100561,
  99172,
  95793,
  98462,
  96280,
  9616,
  104451,
  7313,
  1981,
  198,
  99250,
  103839,
  104981,
  108830,
  99215,
  3709,
  96442,
  97071,
  98150,
  97237,
  101632,
  97553,
  105718,
  3709,
  112235,
  109444,
  99931,
  1710,
  271,
  550,
  1981,
  220,
  97319,
  98144,
  97995,
  9616,
  97240,
  100086,
  7313,
  1981,
  198,
  111083,
  97149,
  97155,
  100899,
  100593,
  332,
  121429,
  97237,
  99486,
  332,
  95726,
  102330,
  3709,
  98118,
  332,
  96725,
  100627,
  96125,
  5202,
  99462,
  332,
  3709,
  132826,
  95761,
  96903,
  97019,
  100713,
  271,
  13962,
  220,
  96588,
  99193,
  4960,
  198,
  16,
  13,
  220,
  95933,
  96227,
  97279,
  100593,
  102330,
  3709,
  332,
  147786,
  104062,
  332,
  198,
  17,
  13,
  220,
  98851,
  104062,
  96671,
  97033,
  3709,
  97319,
  332,
  100627,
  116034,
  97447,
  96725,
  98267,
  5202,
  99462,
  332,
  3709,
  9

In [24]:
tokenizer.decode(tokenized_ds["train"][2]["input_ids"])

'<|im_start|>user\n## === 你的角色描述和背景信息（仅供参考） ===\n你是家居博览会的客服，需要联系报名用户确认是否参会，并提供展会详情。\n\n## === 你的核心任务（必须遵守） ===\n你需要根据以下指示判断**最后一次用户输入**的意图，然后**仅输出一个JSON对象**，绝对不能是其他任何格式\n\n### 重要规则：\n1. 你只负责判断意图，**不进行对话**\n2. 无论对话历史如何，你的**输出必须是且仅是一个JSON对象**，不得输出为空、不得输出其他格式的数据\n3. 禁止输出任何自然语言、解释或额外内容\n4. 如果之前的回复是自然语言，忽略它并继续输出JSON\n5. JSON必须包含且仅包含两个字段：`input_summary`、`intention_id`。字段值必须为字符串，格式必须严格遵循：{\'input_summary\': \'...\', \'intention_id\': \'...\'}\n\n### JSON字段说明：\n1. **input_summary**\n- 务必参考**智能助手和用户的对话历史**，用不超过10个汉字，清晰简洁地概括**最后一次用户输入**的核心内容\n- 示例：“用户有兴趣参加活动”、“用户不想被打扰”、“用户询问你是谁”\n\n2. **intention_id**\n- 务必参考**智能助手和用户的对话历史**，判断**最后一次用户输入**的意图\n- 仅可从以下【意图库列表】或【知识库列表】中，根据**意图名称**和**意图说明**，选择**唯一最匹配**的意图\n- 若最后一次用户输入无法匹配任一意图，输出 `intention_id` 为 \'others\'\n- 如果最后一次用户输入的语义匹配某一条意图的**意图名称**和**意图说明**，则将这条意图的**意图id**输出为 `intention_id`\n- 禁止自行创建或修改 `intention_id`，其值必须严格来自两个列表中的意图id，或为 \'others\'\n- 选择优先级说明：\n  - 如果两个列表均有内容，首先使用【意图库列表】判别用户的意图，当匹配成功，将**意图id**输出为 `intention_id`；若无法匹配，才使用【知识库列表】判别用户的意图，当匹配成功

In [25]:
tokenizer.decode(list(filter(lambda x:x!=-100,tokenized_ds["train"][2]["labels"])))

'{"input_summary": "用户否认报名", "intention_id": "7b2e9f1c3a4d8e60"}<|im_end|>\n<|im_end|>'

In [26]:
tokenized_ds["train"][2]["input_ids"]

[248045,
 846,
 198,
 550,
 1981,
 220,
 97319,
 100561,
 99172,
 95793,
 98462,
 96280,
 9616,
 104451,
 7313,
 1981,
 198,
 99250,
 103839,
 104981,
 108830,
 99215,
 3709,
 96442,
 97071,
 98150,
 97237,
 101632,
 97553,
 105718,
 3709,
 112235,
 109444,
 99931,
 1710,
 271,
 550,
 1981,
 220,
 97319,
 98144,
 97995,
 9616,
 97240,
 100086,
 7313,
 1981,
 198,
 111083,
 97149,
 97155,
 100899,
 100593,
 332,
 121429,
 97237,
 99486,
 332,
 95726,
 102330,
 3709,
 98118,
 332,
 96725,
 100627,
 96125,
 5202,
 99462,
 332,
 3709,
 132826,
 95761,
 96903,
 97019,
 100713,
 271,
 13962,
 220,
 96588,
 99193,
 4960,
 198,
 16,
 13,
 220,
 95933,
 96227,
 97279,
 100593,
 102330,
 3709,
 332,
 147786,
 104062,
 332,
 198,
 17,
 13,
 220,
 98851,
 104062,
 96671,
 97033,
 3709,
 97319,
 332,
 100627,
 116034,
 97447,
 96725,
 98267,
 5202,
 99462,
 332,
 3709,
 97499,
 100627,
 138565,
 5205,
 97499,
 100627,
 96903,
 100713,
 104027,
 198,
 18,
 13,
 220,
 103823,
 100627,
 97019,
 96963,

In [27]:
tokenized_ds["train"][2]["labels"]

[-100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,

In [28]:
len(tokenized_ds["train"][2]["labels"])

1306

## 3. Model LoRA Configuration with Unsloth

In [29]:
sum(param.numel() for param in model.parameters())

4539265536

In [30]:
model.device

device(type='cuda', index=0)

Add LoRA adapter to the model with FastLanguageModel from Unsloth.  
For the 4B model, we use r=32, and alpha=64.

In [31]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",  # unsloth's optimized gradient checkpointing
    random_state=SEED,
    finetune_vision_layers=False,  
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: Making `model.base_model.model.model.language_model` require gradients


In [32]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3_5ForConditionalGeneration(
      (model): Qwen3_5Model(
        (visual): Qwen3_5VisionModel(
          (patch_embed): Qwen3_5VisionPatchEmbed(
            (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
          )
          (pos_embed): Embedding(2304, 1024)
          (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-23): 24 x Qwen3_5VisionBlock(
              (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
              (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
              (attn): Qwen3_5VisionAttention(
                (qkv): Linear(in_features=1024, out_features=3072, bias=True)
                (proj): Linear(in_features=1024, out_features=1024, bias=True)
              )
              (mlp): Qwen3_5VisionMLP(
                (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
               

In [33]:
model.print_trainable_parameters()

trainable params: 42,467,328 || all params: 4,581,732,864 || trainable%: 0.9269


The parameters that need to be adjusts is significantly decreased from 4.5B to 42M.

In [34]:
for name, parameter in model.named_parameters():
    print(name)

base_model.model.model.visual.patch_embed.proj.weight
base_model.model.model.visual.patch_embed.proj.bias
base_model.model.model.visual.pos_embed.weight
base_model.model.model.visual.blocks.0.norm1.weight
base_model.model.model.visual.blocks.0.norm1.bias
base_model.model.model.visual.blocks.0.norm2.weight
base_model.model.model.visual.blocks.0.norm2.bias
base_model.model.model.visual.blocks.0.attn.qkv.weight
base_model.model.model.visual.blocks.0.attn.qkv.bias
base_model.model.model.visual.blocks.0.attn.proj.weight
base_model.model.model.visual.blocks.0.attn.proj.bias
base_model.model.model.visual.blocks.0.mlp.linear_fc1.weight
base_model.model.model.visual.blocks.0.mlp.linear_fc1.bias
base_model.model.model.visual.blocks.0.mlp.linear_fc2.weight
base_model.model.model.visual.blocks.0.mlp.linear_fc2.bias
base_model.model.model.visual.blocks.1.norm1.weight
base_model.model.model.visual.blocks.1.norm1.bias
base_model.model.model.visual.blocks.1.norm2.weight
base_model.model.model.visual.b

## 4. Configure the training

In [35]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR, #to store the prediction results and checkpoints of the model file
    per_device_train_batch_size = 2, #8 by default
    per_device_eval_batch_size = 2,
    gradient_accumulation_steps = 4, # Effective batch size = 8

    num_train_epochs=3,

    learning_rate=1e-4,
    lr_scheduler_type="cosine",

    # ~8000 training samples
    warmup_ratio=0.03,
    weight_decay=0.01,
    
    max_grad_norm=1.0,
    
    # Logging
    logging_steps=10,
    logging_first_step=True,

    # Evaluation
    eval_strategy="steps",
    eval_steps=150, #~8000 training samples

    # Saving
    save_strategy="steps",
    save_steps=150,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Precision
    bf16=True,
    tf32=True,
    
    # gradient_checkpointing handed off to unsloth above
    gradient_checkpointing=False,
    
    # Performance
    dataloader_num_workers=4,
    report_to="none",
    remove_unused_columns=True # Set to True since we cleaned the dataset columns
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [36]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100
)

## 5. Create the trainer and train

In [37]:
trainer = Trainer(
    model = model,
    args = args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator = data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [38]:
trainer.train()

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
150,0.739224,0.147422
300,0.435166,0.121598
450,0.559811,0.125022
600,0.422978,0.120628
750,0.500623,0.121185
900,0.575499,0.113095
1050,0.424945,0.112691
1200,0.292299,0.117000
1350,0.309175,0.112261
1500,0.331130,0.114996


Unsloth: Not an error, but Qwen3_5ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/root/miniconda3/lib/python3.12/multiprocessing/util.py", line 303, in _run_finalizers
    finalizer()
  File "/root/miniconda3/lib/python3.12/multiprocessing/util.py", line 303, in _run_finalizers
    finalizer()
  File "/root/miniconda3/lib/python3.12/multiprocessing/util.py", line 227, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/miniconda3/lib/python3.12/multiprocessing/util.py", line 303, in _run_finalizers
    finalizer()
  File "/root/miniconda3/lib/python3.12/multiprocessing/util.py", line 227, in __call__


TrainOutput(global_step=2250, training_loss=0.41017535124884713, metrics={'train_runtime': 20270.8055, 'train_samples_per_second': 1.122, 'train_steps_per_second': 0.14, 'total_flos': 6.4161890919791e+17, 'train_loss': 0.41017535124884713, 'epoch': 2.3737133808392716})

## 7. Save model

In [39]:
# Create output directory with timestamp
adapter_dir = f"./lora_adapter_lr1e-4_3epoch"

In [40]:
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print("Model saved successfully!")

Model saved successfully!
